# Preliminary Analysis

Defines the parameters used by `02_regression`: noise level, best feature
representation, and PCA component count per sensor.

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.manifold import TSNE
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 10,
    'figure.figsize': (10, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Constants
FILE_PATH       = 'processed_data.pkl'
RANDOM_STATE    = 42
PRELIMINARY_OUT = 'preliminary_results.pkl'

# Cross-sensor comparison settings (exploratory visualisation only)
COMMON_REPR = 'real_imag'   # None -> each sensor uses its ablation winner
COMMON_COMP = 20            # None -> each sensor uses its scree-derived count

## Load Data & Noise Floor

In [ ]:
try:
    df = pd.read_pickle(FILE_PATH)
    print(f"Loaded '{FILE_PATH}': {len(df)} sweeps across "
          f"{df['target_class_date'].nunique()} best-before classes.")
except FileNotFoundError:
    raise FileNotFoundError(f"'{FILE_PATH}' not found. Run 02_loading_v2.ipynb first.")

# Noise floor from within-session repeated sweeps
df_sorted = df.sort_values(['filename', 'sweep_id']).copy()
noise_estimates_a, noise_estimates_b = [], []

for fname, grp in df_sorted.groupby('filename'):
    if len(grp) >= 5:
        sweeps_a = np.stack(grp['features_a'].values)
        sweeps_b = np.stack(grp['features_b'].values)
        noise_estimates_a.append(sweeps_a.std(axis=0))
        noise_estimates_b.append(sweeps_b.std(axis=0))

if not noise_estimates_a:
    print("WARNING: No session with ≥5 sweeps found. Falling back to NOISE_LEVEL = 0.0075")
    NOISE_LEVEL = 0.0035
else:
    mean_noise_a = np.mean(np.stack(noise_estimates_a), axis=0)
    mean_noise_b = np.mean(np.stack(noise_estimates_b), axis=0)
    empirical_noise_a = float(np.median(mean_noise_a))
    empirical_noise_b = float(np.median(mean_noise_b))
    NOISE_LEVEL = round(float(np.mean([empirical_noise_a, empirical_noise_b])), 6)

    print(f"Noise floor — Sensor A (S11): {empirical_noise_a:.6f}")
    print(f"Noise floor — Sensor B (S22): {empirical_noise_b:.6f}")
    print(f"Adopted NOISE_LEVEL          : {NOISE_LEVEL:.6f}")

    num_freqs   = mean_noise_a.shape[0] // 2
    frequencies = np.linspace(0.001, 6.0, num_freqs)
    fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
    fig.suptitle('Empirical VNA Noise Floor (per-frequency std across repeated sweeps)', fontsize=13)
    for ax, noise, label in zip(
            axes.flat,
            [mean_noise_a[:num_freqs], mean_noise_a[num_freqs:],
             mean_noise_b[:num_freqs], mean_noise_b[num_freqs:]],
            ['S11 Real', 'S11 Imaginary', 'S22 Real', 'S22 Imaginary']):
        ax.plot(frequencies, noise, lw=1, color='steelblue')
        ax.axhline(NOISE_LEVEL, color='crimson', ls='--', lw=1.2,
                   label=f'Adopted σ = {NOISE_LEVEL:.4f}')
        ax.set_title(label)
        ax.set_ylabel('Std Dev')
        ax.legend(fontsize=8)
        ax.grid(True, linestyle='--', alpha=0.4)
    for ax in axes[1]:
        ax.set_xlabel('Frequency (GHz)')
    plt.tight_layout()
    plt.show()

## Feature Representation Ablation

Compares three representations of the complex S-parameter data with a PCA+SVR
probe fitted on all data. In-sample R² ranks them by how much shelf-life
signal each captures, not by generalisation.

In [ ]:
def build_feature_representations(X_raw: np.ndarray) -> dict:
    n    = X_raw.shape[1] // 2
    real = X_raw[:, :n]
    imag = X_raw[:, n:]
    return {
        'real_imag': X_raw,
        'magnitude': np.sqrt(real**2 + imag**2),
        'phase':     np.arctan2(imag, real),
    }


def ablate_representations(df, feature_col, sensor_name, n_components=20):
    """
    Compares feature representations by fitting a PCA + SVR probe on all data
    and reporting in-sample R².
    Returns (results_dict, best_repr_name).
    """
    print(f"\n{'─'*60}")
    print(f"  Ablation: {sensor_name}  (probe n_components={n_components})")
    print(f"{'─'*60}")

    X_raw = np.stack(df[feature_col].values)
    y     = df['timedelta_days'].values
    reprs = build_feature_representations(X_raw)

    results = {}
    for repr_name, X in reprs.items():
        X_noisy = X + np.random.normal(0, NOISE_LEVEL, X.shape)
        pipe = Pipeline([
            ('scaler',  StandardScaler()),
            ('reducer', PCA(n_components=n_components, random_state=RANDOM_STATE)),
            ('reg',     SVR(kernel='rbf')),
        ])
        pipe.fit(X_noisy, y)
        r2 = r2_score(y, pipe.predict(X_noisy))  # in-sample
        results[repr_name] = r2
        print(f"  {repr_name:12s}  in-sample R² = {r2:.4f}")

    best_repr = max(results, key=results.__getitem__)
    print(f"\n  ✓ Best representation for {sensor_name}: '{best_repr}'")
    return results, best_repr


PROBE_COMPONENTS = 20
ablation_a, best_repr_a = ablate_representations(df, 'features_a', 'Sensor A (S11)', PROBE_COMPONENTS)
ablation_b, best_repr_b = ablate_representations(df, 'features_b', 'Sensor B (S22)', PROBE_COMPONENTS)

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle('Feature Representation Ablation (PCA+SVR, in-sample R²)', fontsize=13)
repr_labels = ['real_imag', 'magnitude', 'phase']
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, ablation, best, title in zip(
        axes,
        [ablation_a, ablation_b],
        [best_repr_a, best_repr_b],
        ['Sensor A (S11)', 'Sensor B (S22)']):
    vals = [ablation[r] for r in repr_labels]
    bars = ax.bar(repr_labels, vals, color=colors, edgecolor='black', alpha=0.85)
    for bar, label in zip(bars, repr_labels):
        if label == best:
            bar.set_edgecolor('crimson')
            bar.set_linewidth(2.5)
    ax.set_title(title)
    ax.set_ylabel('In-sample R²')
    ax.set_xlabel('Feature Representation')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.annotate(f'Best: {best}', xy=(0.5, 0.95), xycoords='axes fraction',
                ha='center', fontsize=9, color='crimson')

plt.tight_layout()
plt.show()
print(f"\nAdopted → Sensor A: '{best_repr_a}',  Sensor B: '{best_repr_b}'")

## PCA Dimensionality Selection

PCA fitted on the full dataset per sensor. The component count is the 95% EVR
threshold, past which extra components add little variance.

In [ ]:
def select_n_components(df, feature_col, best_repr, sensor_name, max_components=40):
    """
    Fits PCA on full data and selects n_components at the 95% EVR threshold.
    Returns (n_95, cumulative_evr, evr_per_component).
    """
    print(f"\n{'─'*60}")
    print(f"  PCA selection: {sensor_name}  repr='{best_repr}'")
    print(f"{'─'*60}")

    X_raw = np.stack(df[feature_col].values)
    X     = build_feature_representations(X_raw)[best_repr]
    y     = df['timedelta_days'].values

    X_noisy = X #+ np.random.normal(0, NOISE_LEVEL, X.shape)
    scaler  = StandardScaler().fit(X_noisy)
    X_sc    = scaler.transform(X_noisy)

    max_components = min(max_components, X_sc.shape[1], X_sc.shape[0] - 1)
    pca = PCA(n_components=max_components, random_state=RANDOM_STATE)
    pca.fit(X_sc)

    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_95   = int(np.searchsorted(cumvar, 0.95)) + 1
    print(f"  95% EVR threshold: {n_95} components")

    # Plot
    comp_x = np.arange(1, max_components + 1)
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.suptitle(f'PCA Scree — {sensor_name}  (repr: {best_repr})', fontsize=12)

    ax.bar(comp_x, pca.explained_variance_ratio_ * 100,
           color='steelblue', alpha=0.7, label='Individual EVR')
    ax2 = ax.twinx()
    ax2.plot(comp_x, cumvar * 100, 'o-', color='crimson', ms=3, lw=1.5,
             label='Cumulative EVR')
    ax2.axhline(95, ls='--', color='grey', lw=1)
    ax2.axvline(n_95, ls='--', color='grey', lw=1, label=f'95% EVR = {n_95}')
    ax2.set_ylabel('Cumulative EVR (%)')
    ax2.set_ylim(0, 105)
    ax2.legend(fontsize=8, loc='center right')
    ax.set_xlabel('Principal Component')
    ax.set_ylabel('Individual EVR (%)')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

    return n_95, cumvar, pca.explained_variance_ratio_


n_comp_a, cumvar_a, evr_a = select_n_components(df, 'features_a', best_repr_a, 'Sensor A (S11)')
n_comp_b, cumvar_b, evr_b = select_n_components(df, 'features_b', best_repr_b, 'Sensor B (S22)')

print(f"\n→  Sensor A: repr='{best_repr_a}', N_COMPONENTS={n_comp_a}")
print(f"   Sensor B: repr='{best_repr_b}', N_COMPONENTS={n_comp_b}")

## Exploratory Visualisation

PCA, PLS, and t-SNE projections fitted on all data. A qualitative check of
whether the signal separates in low dimensions.

In [ ]:
def knn_preservation_score(X_high, X_low, k=10):
    """Fraction of k-NN in high-dim space that are also k-NN in the embedding."""
    k = min(k, len(X_high) - 1)
    nbrs_h = NearestNeighbors(n_neighbors=k).fit(X_high).kneighbors(return_distance=False)
    nbrs_l = NearestNeighbors(n_neighbors=k).fit(X_low).kneighbors(return_distance=False)
    return float(np.mean([
        len(set(nbrs_h[i]) & set(nbrs_l[i])) / k for i in range(len(X_high))
    ]))


def select_tsne_perplexity(X_scaled, candidate_perplexities=None, k_knn=10):
    """Sweeps perplexity values; selects the one maximising KNN preservation."""
    n = len(X_scaled)
    if candidate_perplexities is None:
        candidate_perplexities = [p for p in [5, 10, 20, 30, 50] if p < n]
    print(f"  Sweeping perplexity ∈ {candidate_perplexities}  (n={n})")
    scores, embeddings = {}, {}
    for perp in candidate_perplexities:
        emb = TSNE(n_components=2, perplexity=perp,
                   random_state=RANDOM_STATE, max_iter=1000).fit_transform(X_scaled)
        scores[perp]     = knn_preservation_score(X_scaled, emb, k=k_knn)
        embeddings[perp] = emb
        print(f"    perplexity={perp:>3d}  KNN preservation = {scores[perp]:.4f}")
    best_perp = max(scores, key=scores.__getitem__)
    print(f"  ✓ Best perplexity: {best_perp}  (score = {scores[best_perp]:.4f})")
    return best_perp, scores, embeddings

In [ ]:
def explore_sensor_data(df, feature_col, sensor_name, best_repr, n_components):
    print(f"\n{'='*30} EXPLORATORY ANALYSIS: {sensor_name} {'='*30}")

    X_raw = np.stack(df[feature_col].values)
    y     = df['timedelta_days'].values
    X     = build_feature_representations(X_raw)[best_repr]

    X_noisy = X + np.random.normal(0, NOISE_LEVEL, X.shape)
    scaler  = StandardScaler().fit(X_noisy)
    X_sc    = scaler.transform(X_noisy)

    n_raw_freqs = X_raw.shape[1] // 2
    frequencies = np.linspace(0.001, 6.0, n_raw_freqs)

    # PCA
    print(f"\n--- PCA (n_components={n_components}) ---")
    pca   = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_sc)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'PCA — {sensor_name}  (repr: {best_repr})', fontsize=13)
    sc = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                         c=y, cmap='viridis', s=30, alpha=0.8, edgecolors='none')
    plt.colorbar(sc, ax=axes[0], label='Days to Best-Before')
    axes[0].set_title('PC1 vs PC2')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    axes[0].grid(True, linestyle='--', alpha=0.4)

    if best_repr == 'real_imag':
        window = max(3, n_raw_freqs // 50)
        kernel = np.ones(window) / window
        for i in range(min(3, n_components)):
            axes[1].plot(frequencies,
                         np.convolve(pca.components_[i, :n_raw_freqs], kernel, mode='same'),
                         label=f'PC{i+1}')
        axes[1].set_ylabel('Loading Weight (Real, smoothed)')
    else:
        for i in range(min(3, n_components)):
            axes[1].plot(frequencies, pca.components_[i], label=f'PC{i+1}')
        axes[1].set_ylabel('Loading Weight')
    axes[1].set_xlabel('Frequency (GHz)')
    axes[1].set_title('PC Loadings')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

    # PLSR
    print("\n--- PLS Projection ---")
    pls        = PLSRegression(n_components=3)
    X_pls, _   = pls.fit_transform(X_sc, y)
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(X_pls[:, 0], X_pls[:, 1],
                    c=y, cmap='plasma', s=30, alpha=0.8, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='Days to Best-Before')
    ax.set_title(f'PLS 2D Projection — {sensor_name}')
    ax.set_xlabel('PLSC 1')
    ax.set_ylabel('PLSC 2')
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

    # t-SNE
    print("\n--- t-SNE ---")
    X_for_tsne = X_pca[:, :min(50, n_components)]
    best_perp, perp_scores, perp_embeddings = select_tsne_perplexity(X_for_tsne)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f't-SNE — {sensor_name}', fontsize=13)
    perps = sorted(perp_scores.keys())
    axes[0].plot(perps, [perp_scores[p] for p in perps], 'o-', color='steelblue', ms=6)
    axes[0].axvline(best_perp, ls='--', color='crimson', lw=1.5,
                    label=f'Best = {best_perp}')
    axes[0].set_xlabel('Perplexity')
    axes[0].set_ylabel('KNN Preservation')
    axes[0].set_title('Perplexity Selection')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.4)
    sc = axes[1].scatter(perp_embeddings[best_perp][:, 0],
                         perp_embeddings[best_perp][:, 1],
                         c=y, cmap='magma', s=30, alpha=0.8, edgecolors='none')
    plt.colorbar(sc, ax=axes[1], label='Days to Best-Before')
    axes[1].set_title(f't-SNE (perplexity={best_perp})')
    axes[1].set_xlabel('t-SNE Dim 1')
    axes[1].set_ylabel('t-SNE Dim 2')
    axes[1].grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

    print('─' * 75)
    return best_perp


# Resolve common settings
repr_a = COMMON_REPR if COMMON_REPR is not None else best_repr_a
comp_a = COMMON_COMP if COMMON_COMP is not None else n_comp_a
repr_b = COMMON_REPR if COMMON_REPR is not None else best_repr_b
comp_b = COMMON_COMP if COMMON_COMP is not None else n_comp_b

if COMMON_REPR is not None or COMMON_COMP is not None:
    print(f"Exploratory plots: repr={repr_a!r}, components={comp_a}")
    print(f"  (Sensor A optimal: repr='{best_repr_a}', n={n_comp_a}  |  "
          f"Sensor B: repr='{best_repr_b}', n={n_comp_b})")
else:
    print("Exploratory plots using per-sensor optimal settings.")

best_perp_a = explore_sensor_data(df, 'features_a', 'Sensor A (S11)', repr_a, comp_a)
best_perp_b = explore_sensor_data(df, 'features_b', 'Sensor B (S22)', repr_b, comp_b)

## Export Results

In [ ]:
prelim = {
    'noise_level': NOISE_LEVEL,
    'sensor_a': {
        'best_repr':            best_repr_a,
        'n_components':         n_comp_a,
        'best_tsne_perplexity': best_perp_a,
        'cumulative_evr':       cumvar_a,
        'evr_per_component':    evr_a,
        'ablation_r2':          ablation_a,   # in-sample R^2 pesr representation
    },
    'sensor_b': {
        'best_repr':            best_repr_b,
        'n_components':         n_comp_b,
        'best_tsne_perplexity': best_perp_b,
        'cumulative_evr':       cumvar_b,
        'evr_per_component':    evr_b,
        'ablation_r2':          ablation_b,
    },
    # Settings used for Section 4 exploratory plots (comparison only)
    'explore': {
        'repr_a':         repr_a,
        'repr_b':         repr_b,
        'n_components_a': comp_a,
        'n_components_b': comp_b,
    },
}

with open(PRELIMINARY_OUT, 'wb') as f:
    pickle.dump(prelim, f)

print(f"Saved '{PRELIMINARY_OUT}'")
print()
print("Per-sensor optimal values (fed into 02_regression_v3.ipynb):")
print(f"  NOISE_LEVEL : {NOISE_LEVEL}")
print(f"  Sensor A    : repr='{best_repr_a}', N_COMPONENTS={n_comp_a}")
print(f"  Sensor B    : repr='{best_repr_b}', N_COMPONENTS={n_comp_b}")
print(f"  t-SNE perp  : A={best_perp_a}, B={best_perp_b}")
print()
print("Exploratory visualisation settings (Section 4 only):")
print(f"  repr={repr_a!r}, N_COMPONENTS={comp_a}")